# StormEngine V6 — Persistence baselines

Run non-learned references on the same frozen 2017 windows as V6. The dense-grid reference has access to more information than V6; the sparse IDW reference uses the same 390 point coordinates and is the fair comparison.

In [ ]:
from pathlib import Path
import json, subprocess, sys
import matplotlib.pyplot as plt

here = Path.cwd().resolve()
REPO = here if (here / 'pyproject.toml').exists() else here.parent
assert (REPO / 'pyproject.toml').exists(), REPO
CONFIG = REPO / 'configs' / 'era5_2010_2017_windows.local.yaml'
V6_METRICS = REPO / 'results' / 'v6_2010_2017_baseline' / 'metrics_by_lead.json'
OUTPUT_DIR = REPO / 'artifacts' / 'v6_2010_2017' / 'baselines_test'
print('Repository:', REPO)
print('Configuration:', CONFIG)
print('V6 metrics:', V6_METRICS)

## 1. Evaluate both references

This reads the existing hourly memory-mapped cache. It does not train a model and does not require CUDA.

In [ ]:
command = [
    sys.executable, '-u', str(REPO / 'scripts' / 'evaluate_baselines.py'),
    '--config', str(CONFIG), '--split', 'test',
    '--v6-metrics', str(V6_METRICS), '--output-dir', str(OUTPUT_DIR),
]
print('Running:', ' '.join(command))
subprocess.run(command, cwd=REPO, check=True)

## 2. Aggregate RMSE and skill

In [ ]:
result = json.loads((OUTPUT_DIR / 'baseline_metrics.json').read_text(encoding='utf-8'))
variables = result['variables']
for baseline, values in result['baselines'].items():
    print('\n', baseline)
    for region in ('full', 'land', 'sea'):
        metrics = values['metrics']['aggregate'][region]
        summary = ' '.join(f"{name}={metrics[name]['rmse']:.4f}" for name in variables)
        print(f'  {region}: {summary}')
    print('  V6 aggregate skill:')
    for region in ('full', 'land', 'sea'):
        skill = result['v6_skill'][baseline]['aggregate'][region]
        summary = ' '.join(f"{name}={skill[name]['skill']:+.3f}" for name in variables)
        print(f'    {region}: {summary}')

## 3. V6 versus baselines by lead hour

Solid lines are full-domain RMSE; dashed lines are sea-only RMSE.

In [ ]:
v6 = json.loads(V6_METRICS.read_text(encoding='utf-8'))['metrics']['by_lead_hour']
hours = sorted(map(int, v6))
colors = {'V6': 'black', 'dense persistence': 'tab:blue', 'sparse IDW': 'tab:orange'}
sources = {
    'V6': v6,
    'dense persistence': result['baselines']['dense_grid_persistence']['metrics']['by_lead_hour'],
    'sparse IDW': result['baselines']['sparse_idw_persistence']['metrics']['by_lead_hour'],
}
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True)
for variable, axis in zip(variables, axes.flat):
    for label, source in sources.items():
        full = [source[str(hour)]['full'][variable]['rmse'] for hour in hours]
        sea = [source[str(hour)]['sea'][variable]['rmse'] for hour in hours]
        axis.plot(hours, full, color=colors[label], marker='o', label=label)
        axis.plot(hours, sea, color=colors[label], marker='o', linestyle='--', alpha=.65)
    axis.set_title(variable); axis.set_xlabel('Lead hour'); axis.set_ylabel('RMSE'); axis.grid(alpha=.3)
axes.flat[-1].axis('off')
axes.flat[0].legend(fontsize=8)
fig.tight_layout(); plt.show()